# Extracting Metadata from ALSF pilot miscroscopy data

## Overview
In this notebook, we extract and write metadata associated with microscopy images to obtain information including plate row/column, channel, measurement barcode, and time of measurement.  

## Import libraries

In [1]:
import os
import re
import pathlib
import pandas as pd
import xml.etree.ElementTree as ET

## Define data and output location

In [2]:
# Miscroscopy dataset path
DATA_PATH = "/home/newusr/Waylab/ALSF_img2img/data/ALSF_pilot_data/SN0313537/"
# Location to save the extracted metadata information
OUTPUT_PATH = os.path.join(
    pathlib.Path(".").absolute(),
    "metadata"
)
os.makedirs(OUTPUT_PATH, exist_ok=True)

## Define helper functions for XML file parsing

In [3]:
import os
import pandas as pd
import xml.etree.ElementTree as ET

# Helper function to parse FFC XML files to extract ChannelID to ChannelName mappings
def parse_channel_mapping(xml_path: str)-> dict:
    """
    Parse the XML file that lives FFC_Profile/ under to extract ChannelID to ChannelName mapping
    
    :param xml_path: path to the XML file
    :type xml_path: str

    :return: dictionary of channel mappings
    :rtype: dict
    """
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        namespace = {"ns": root.tag.split("}")[0].strip("{")}
        channel_map = {}

        map_element = root.find("ns:Map", namespace)
        if not map_element:
            print(f"No <Map> element found in {xml_path}")
            return channel_map

        for entry in map_element.findall("ns:Entry", namespace):
            channel_id = entry.attrib.get("ChannelID")
            flatfield_profile = entry.find("ns:FlatfieldProfile", namespace)
            if flatfield_profile is not None:
                flatfield_text = flatfield_profile.text or ""
                if "ChannelName:" in flatfield_text:
                    channel_name = flatfield_text.split("ChannelName:")[1].split(",")[0].strip()
                    if channel_id and channel_name:
                        channel_map[channel_id] = channel_name

        return channel_map
    except Exception as e:
        print(f"Error parsing XML at {xml_path}: {e}")
        return {}

# Helper function to parse image index XML files
def parse_image_index(xml_path: str)-> pd.DataFrame:
    """
    Parse the XML file that lives under Images/ to extract image index data.

    :param xml_path: path to the XML file
    :type xml_path: str

    :return: DataFrame with image index data
    :rtype: pd.DataFrame
    """
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        namespace = {"ns": root.tag.split("}")[0].strip("{")}

        # Extract global metadata
        plate_id = root.findtext("ns:Plates/ns:Plate/ns:PlateID", default="", namespaces=namespace)
        measurement_id = root.findtext("ns:Plates/ns:Plate/ns:MeasurementID", default="", namespaces=namespace)
        measurement_start_time = root.findtext("ns:Plates/ns:Plate/ns:MeasurementStartTime", default="", namespaces=namespace)

        # Collect data from <Images>
        images_field = root.find("ns:Images", namespace)
        rows = []
        if images_field is not None:
            for image in images_field.findall("ns:Image", namespace):
                image_data = {
                    "PlateID": plate_id,
                    "MeasurementID": measurement_id,
                    "MeasurementStartTime": measurement_start_time,
                    "Version": image.attrib.get("Version"),
                }
                image_data.update({
                    field.tag.split("}")[-1]: field.text
                    for field in image
                })
                rows.append(image_data)

        return pd.DataFrame(rows)
    except Exception as e:
        print(f"Error processing XML at {xml_path}: {e}")
        return pd.DataFrame()

# Function to walk through the data directory and makes calls to the XML parsing helper functions 
def walk_and_parse_xml(data_root: str):
    """
    Traverse the data directory to extract channel mappings and image index metadata.

    Anticipated directory structure:
    RootDirectory/
    ├── <date>_<cell_line>_<Re-imaged>/  # Type a) directory (re-image status is 'Re-imaged')
    │   ├── <plate_barcode>__<yyyy>-<mm>-<dd>T<hh>_<mm>_<ss>_Measurement<measurement_number>/  # Type b) directory
    │   │   ├── Assaylayout/
    │   │   │   └── AssayLayout.xml
    │   │   ├── FFC_Profile/
    │   │   │   └── FFC_Profile.xml
    │   │   └── Images/
    │   │       ├── Index.xml
    │   │       ├── <image_1>.tiff
    │   │       ├── <image_2>.tiff
    │   │       └── ... (other TIFF files)
    ├── <date>_<cell_line>/  # Type a) directory (re-image status is 'Original')
    │   ├── <plate_barcode>__<yyyy>-<mm>-<dd>T<hh>_<mm>_<ss>_Measurement<measurement_number>/  # Type b) directory
    │   │   ├── Assaylayout/
    │   │   │   └── AssayLayout.xml
    │   │   ├── FFC_Profile/
    │   │   │   └── FFC_Profile.xml
    │   │   └── Images/
    │   │       ├── Index.xml
    │   │       ├── <image_1>.tiff
    │   │       ├── <image_2>.tiff
    │   │       └── ... (other TIFF files)
    └── ... (other type a) directories)

    :param data_root: path to the root directory of the dataset
    :type data_root: str

    :return: tuple, DataFrame with channel mappings and DataFrame with image index data
    :rtype: pd.DataFrame, pd.DataFrame
    """
    ffc_rows = []
    df_index = pd.DataFrame()

    for root, dirs, _ in os.walk(data_root):
        # Process FFC_Profile directories
        if "FFC_Profile" in dirs:
            ffc_profile_path = os.path.join(root, "FFC_Profile")
            xml_files = [f for f in os.listdir(ffc_profile_path) if f.endswith(".xml")]
            if xml_files:
                plate_barcode = os.path.basename(root).split("__")[0]
                xml_path = os.path.join(ffc_profile_path, xml_files[0])
                channel_mapping = parse_channel_mapping(xml_path)

                # Prepare row with channel mappings
                row = {"PlateID": plate_barcode} # define the same fieldname as in the image index
                row.update({f"channel_{i}": channel_mapping.get(str(i)) for i in range(1, 7)})
                ffc_rows.append(row)

        # Process Images directories
        if "Images" in dirs:
            images_path = os.path.join(root, "Images")
            xml_files = [f for f in os.listdir(images_path) if f.endswith(".xml")]
            if xml_files:
                xml_path = os.path.join(images_path, xml_files[0])
                df_index = pd.concat([df_index, parse_image_index(xml_path)], ignore_index=True)

    # Convert channel mappings to a DataFrame
    df_ffc = pd.DataFrame(ffc_rows)
    return df_ffc, df_index


## Extract information from XML

In [4]:
df_ffc, df_index = walk_and_parse_xml(DATA_PATH)

In [5]:
df_ffc.head()

,PlateID,channel_1,channel_2,channel_3,channel_4,channel_5,channel_6
0,BR00143978,Brightfield high,Alexa 488,Alexa 647,Alexa 568,Alexa 488 Long (CP),HOECHST 33342
1,BR00143980,Brightfield high,Alexa 488,Alexa 647,Alexa 568,Alexa 488 Long (CP),HOECHST 33342
2,BR00143976,Brightfield high,Alexa 488,Alexa 647,Alexa 568,Alexa 488 Long (CP),HOECHST 33342
3,BR00143977,Brightfield high,Alexa 488,Alexa 647,Alexa 568,Alexa 488 Long (CP),HOECHST 33342
4,BR00143981,Brightfield high,Alexa 488,Alexa 647,Alexa 568,Alexa 488 Long (CP),HOECHST 33342


In [6]:
df_index.head()

,PlateID,MeasurementID,MeasurementStartTime,Version,id,State,URL,Row,Col,FieldID,...,SequenceID,GroupID,ChannelID,FlimID,PositionX,PositionY,PositionZ,AbsPositionZ,MeasurementTimeOffset,AbsTime
0,BR00143978,57ddb76f-2d5b-4904-8743-bb6bafc8a296,2024-07-17T18:48:48.3580223-04:00,1,0314K1F1P1R1,Ok,r03c14f01p01-ch1sk1fk1fl1.tiff,3,14,1,...,1,1,1,1,0,0,0,0.134396702,0,2024-07-17T18:50:48.9006008-04:00
1,BR00143978,57ddb76f-2d5b-4904-8743-bb6bafc8a296,2024-07-17T18:48:48.3580223-04:00,1,0314K1F1P1R2,Ok,r03c14f01p01-ch2sk1fk1fl1.tiff,3,14,1,...,1,1,2,1,0,0,-5E-06,0.1343918,0,2024-07-17T18:50:49.1970008-04:00
2,BR00143978,57ddb76f-2d5b-4904-8743-bb6bafc8a296,2024-07-17T18:48:48.3580223-04:00,1,0314K1F1P1R3,Ok,r03c14f01p01-ch3sk1fk1fl1.tiff,3,14,1,...,1,1,3,1,0,0,-5E-06,0.1343918,0,2024-07-17T18:50:49.3218008-04:00
3,BR00143978,57ddb76f-2d5b-4904-8743-bb6bafc8a296,2024-07-17T18:48:48.3580223-04:00,1,0314K1F1P1R4,Ok,r03c14f01p01-ch4sk1fk1fl1.tiff,3,14,1,...,1,1,4,1,0,0,-3E-06,0.134393796,0,2024-07-17T18:50:49.4466008-04:00
4,BR00143978,57ddb76f-2d5b-4904-8743-bb6bafc8a296,2024-07-17T18:48:48.3580223-04:00,1,0314K1F1P1R5,Ok,r03c14f01p01-ch5sk1fk1fl1.tiff,3,14,1,...,1,1,5,1,0,0,-3E-06,0.134393796,0,2024-07-17T18:50:49.7586008-04:00


## Helper function for determining re-image status

In [7]:
def walk_and_determine_reimage_status(data_root: str):
    """
    Walk through the microscopy data file structure and extract metadata for each TIFF image.

    Anticipated directory structure:
    RootDirectory/
    ├── <date>_<cell_line>_<Re-imaged>/  # Type A) directory (re-image status is 'Re-imaged')
    │   ├── <plate_barcode>__<yyyy>-<mm>-<dd>T<hh>_<mm>_<ss>_Measurement<measurement_number>/  # Type b) directory
    │   │   ├── Assaylayout/
    │   │   │   └── AssayLayout.xml
    │   │   ├── FFC_Profile/
    │   │   │   └── FFC_Profile.xml
    │   │   └── Images/
    │   │       ├── Index.xml
    │   │       ├── <image_1>.tiff
    │   │       ├── <image_2>.tiff
    │   │       └── ... (other TIFF files)
    ├── <date>_<cell_line>/  # Type A) directory (re-image status is 'Original')
    │   ├── <plate_barcode>__<yyyy>-<mm>-<dd>T<hh>_<mm>_<ss>_Measurement<measurement_number>/  # Type B) directory
    │   │   ├── Assaylayout/
    │   │   │   └── AssayLayout.xml
    │   │   ├── FFC_Profile/
    │   │   │   └── FFC_Profile.xml
    │   │   └── Images/
    │   │       ├── Index.xml
    │   │       ├── <image_1>.tiff
    │   │       ├── <image_2>.tiff
    │   │       └── ... (other TIFF files)
    └── ... (other type a) directories)

    :param data_root: path to the root directory of the dataset
    :type data_root: str

    :return: DataFrame with TIFF metadata
    :rtype: pd.DataFrame
    """
    tiff_data = []
    regex_pattern = r'(BR\d{8})__(\d{4}-\d{2}-\d{2})T.*Measurement (\d+)'

    for root, dirs, files in os.walk(data_root):
        # Check if this is a Type B) directory
        match = re.match(regex_pattern, os.path.basename(root))
        if match:
            # Extract information from regex match
            barcode, date, measurement = match.groups()

            # Determine re-image status based on the parent directory name
            parent_dir = os.path.basename(os.path.dirname(root))
            re_image_status = True if "Re-imaged" in parent_dir else False

            # Look for TIFF files in the Images subdirectory
            if "Images" in dirs:
                images_path = os.path.join(root, "Images")
                tiff_files = [f for f in os.listdir(images_path) if f.endswith(".tiff")]

                for tiff in tiff_files:
                    tiff_data.append({
                        "re_imaged": re_image_status,
                        "PlateID": barcode,
                        "Date": date,
                        "Measurement": measurement,
                        "tiff_path": os.path.abspath(os.path.join(images_path, tiff)),
                        "URL": tiff
                    })

    return pd.DataFrame(tiff_data)


In [8]:
df_re_image = walk_and_determine_reimage_status(DATA_PATH)
df_re_image.head()

,re_imaged,PlateID,Date,Measurement,tiff_path,URL
0,True,BR00143978,2024-07-17,2,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,r04c22f03p01-ch6sk1fk1fl1.tiff
1,True,BR00143978,2024-07-17,2,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,r03c21f03p01-ch4sk1fk1fl1.tiff
2,True,BR00143978,2024-07-17,2,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,r03c17f07p01-ch4sk1fk1fl1.tiff
3,True,BR00143978,2024-07-17,2,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,r04c17f05p01-ch5sk1fk1fl1.tiff
4,True,BR00143978,2024-07-17,2,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,r04c20f02p01-ch6sk1fk1fl1.tiff


## Export metadata files

In [9]:
df_ffc.to_csv(os.path.join(OUTPUT_PATH, "channel_mappings.csv"), index=False)
df_index.to_csv(os.path.join(OUTPUT_PATH, "image_index.csv"), index=False)
df_re_image.to_csv(os.path.join(OUTPUT_PATH, "re_image_status.csv"), index=False)